# Chapter 8: Reasoning

**Level 2: Modeling & Querying** · ⏱ about 2.5 hours · Dataset: *Recipes & Nutrition* + the Chapter 7 ontology

### What you will learn

1. How Stardog's **query-time reasoning** works, and how to switch it on per query with a chosen model
2. **RDFS inferences**: types from subclasses, domains and ranges, labels from sub-properties
3. **OWL inferences**: inverse and transitive properties, property chains, **defined classes**
4. **Stardog rules** for what OWL can't express: thresholds, computed values, rules that build on rules
5. What rules **can't** do, and why "vegan" needs a query
6. **Explaining** an inference: the proof behind every inferred fact
7. **Consistency**: catching contradictions with disjoint classes
8. The **cost** of reasoning, seen in query plans and timings

In [1]:
import sys
import time
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import stardog
from rdflib import RDF

from kg import client, models, reasoning, sparql
from kg.recipes import ALLERGEN, CATEGORY, INGREDIENT, REC, RECIPE

pd.set_option('display.max_colwidth', 90)
conn = client.connect()
PREFIX = 'urn:tutorial:ch08'
DATA, MODEL = f'{PREFIX}:data', f'{PREFIX}:model'
MODEL_NAME = 'RecipeKG'
ROOT = Path('..')

with client.transaction(conn) as stats:
    for g in (DATA, MODEL):
        conn.clear(graph_uri=g)
    conn.add(stardog.content.File(str(ROOT / 'data' / 'recipes' / 'recipes.ttl')), graph_uri=DATA)
    conn.add(stardog.content.File(str(ROOT / 'ontology' / 'recipes.ttl')), graph_uri=MODEL)          # Chapter 7
    conn.add(stardog.content.File(str(ROOT / 'ontology' / 'recipes_rules.ttl')), graph_uri=MODEL)    # this chapter
models.register_model(MODEL_NAME, MODEL)
print('Loaded:', stats, '| model', MODEL_NAME, '->', MODEL)

PREFIXES = '''PREFIX rec: <http://example.org/recipes#>
PREFIX recipe: <http://example.org/recipes/recipe/>
PREFIX ingredient: <http://example.org/recipes/ingredient/>
PREFIX category: <http://example.org/recipes/category/>
PREFIX allergen: <http://example.org/recipes/allergen/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
'''
def q(query, reasoning=False, **params):
    '''Query the data graph, optionally reasoning with the RecipeKG model.'''
    return sparql.run_query(conn, PREFIXES + query, graphs=DATA, reasoning=reasoning,
                            schema=MODEL_NAME if reasoning else None, **params)

def both(query, **params):
    '''Run a query without and with reasoning, side by side.'''
    return {'without reasoning': q(query, **params), 'with reasoning': q(query, reasoning=True, **params)}

def short(s):
    return s.rsplit('/', 1)[-1].rsplit('#', 1)[-1]

Loaded: {'added': 1237, 'removed': 0} | model RecipeKG -> urn:tutorial:ch08:model


The model graph now holds the Chapter 7 ontology **plus** a rules file, `ontology/recipes_rules.ttl` (section 8.4). As in Chapter 7, it's registered as the named model `RecipeKG`.

## 8.1 How Stardog reasons

- The graph holds **asserted** triples: what you loaded.
- The model holds **axioms**: subclasses, domains, chains, rules, …
- With reasoning on, a query also returns **inferred** triples: what follows from the data plus the axioms.

Stardog reasons **at query time**. It **rewrites** your query so it also matches everything the model implies, and it never writes inferred triples to disk. So:

- inferences are always up to date: change the data or the model, and the next query sees it;
- the cost is paid on every query that uses reasoning (8.8).

Reasoning is off by default. Switch it on **per query**, and name the model:

```python
conn.select(query, reasoning=True, schema='RecipeKG')
sparql.run_query(conn, query, graphs=…, reasoning=True, schema='RecipeKG')
```

Stardog Studio and Explorer have a **Reasoning** toggle and a **model** selector that do the same.

A first overview: instances per class, asserted vs visible with reasoning:

In [2]:
COUNT_TYPES = 'SELECT ?class (COUNT(DISTINCT ?s) AS ?n) { ?s a ?class  FILTER(STRSTARTS(STR(?class), STR(rec:))) } GROUP BY ?class'
asserted = q(COUNT_TYPES).set_index('class').n.rename('asserted')
inferred = q(COUNT_TYPES, reasoning=True).set_index('class').n.rename('with reasoning')
overview = pd.concat([asserted, inferred], axis=1).fillna(0).astype(int)
overview.index = overview.index.map(short)
overview.sort_values('with reasoning', ascending=False)

,asserted,with reasoning
class,,
IngredientLine,84,84
Ingredient,42,42
Category,28,28
Recipe,14,14
Allergen,14,14
HighProteinRecipe,0,10
HighProteinIngredient,0,9
DairyRecipe,0,6
QuickRecipe,0,5


Five classes appear that **nobody stored** a single instance of: two from OWL definitions (`DairyRecipe`, `SeafoodRecipe`) and three from rules (`QuickRecipe`, `HighProteinRecipe`, `HighProteinIngredient`). The rest of the chapter explains each one.

## 8.2 RDFS inferences

### Sub-classes and sub-properties

In [3]:
r = both('SELECT (COUNT(?c) AS ?concepts) { ?c a skos:Concept }')
print('skos:Concept instances:', r['without reasoning'].concepts[0], '→', r['with reasoning'].concepts[0])

r = both('SELECT ?label { recipe:pad-thai rdfs:label ?label }')
print('rdfs:label of Pad Thai:', r['without reasoning'].label.tolist(), '→', r['with reasoning'].label.tolist())

skos:Concept instances: 0 → 28


rdfs:label of Pad Thai: [] → ['Pad Thai', 'Pad thaï']


- `rec:Category rdfs:subClassOf skos:Concept`, so every category is a SKOS concept, and SKOS-aware tools understand our taxonomy.
- `rec:name rdfs:subPropertyOf rdfs:label`, so every name is also a label, which is what generic tools display.

### Domain and range

Domain and range **add types**. A resource we never typed:

In [4]:
conn.update(PREFIXES + f'INSERT DATA {{ GRAPH <{DATA}> {{ recipe:mystery-dish rec:prepMinutes 5 . }} }}')

r = both('SELECT ?type { recipe:mystery-dish a ?type }')
print('types without reasoning:', r['without reasoning'].type.map(short).tolist())
print('types with reasoning:   ', sorted(r['with reasoning'].type.map(short)))

types without reasoning: []
types with reasoning:    ['QuickRecipe', 'Recipe', 'Thing']


From one triple `recipe:mystery-dish rec:prepMinutes 5`:

1. `rec:prepMinutes rdfs:domain rec:Recipe` → it's a **Recipe**.
2. It's a recipe with 5 minutes' prep → the rule in 8.4 makes it a **QuickRecipe**.
3. Everything is an `owl:Thing`, the top class of OWL.

That's the open world assumption at work (Chapter 1): no error, just a conclusion. It also shows why a **wrong** domain is dangerous. Had `rec:name` been given the domain `rec:Recipe`, every ingredient, category and allergen would now be a recipe too.

In [5]:
conn.update(PREFIXES + f'DELETE DATA {{ GRAPH <{DATA}> {{ recipe:mystery-dish rec:prepMinutes 5 . }} }}')

## 8.3 OWL inferences

### Inverse properties

Only `rec:hasIngredient` is stored. Its inverse `rec:isIngredientOf` works in both directions with reasoning:

In [6]:
r = both('SELECT ?recipe { ingredient:egg rec:isIngredientOf ?r . ?r rec:name ?recipe  FILTER(lang(?recipe) = "en") }')
print('without:', r['without reasoning'].recipe.tolist())
print('with:   ', r['with reasoning'].recipe.tolist())

without: []
with:    ['Chocolate Brownies', 'Crispy Chicken Tenders', 'Pad Thai', 'Pancakes', 'Spaghetti Carbonara']


### Transitive properties

Only direct parents are stored (`skos:broader`). `skos:broaderTransitive` is declared transitive, and `broader` is its sub-property:

In [7]:
r = both('SELECT ?ancestor { category:cheese skos:broaderTransitive/rec:name ?ancestor }')
print('Cheese is under:', r['without reasoning'].ancestor.tolist(), '→', r['with reasoning'].ancestor.tolist())

Cheese is under: [] → ['Dairy', 'Food']


Compare Chapter 6: there we wrote the path `skos:broader+` in every query. Now the **model** says the hierarchy is transitive, once, and every query benefits.

### Property chains

`rec:containsAllergen = rec:hasIngredient ∘ rec:hasAllergen`:

In [8]:
q('''SELECT ?recipe (GROUP_CONCAT(DISTINCT ?a; separator=", ") AS ?allergens)
WHERE { ?r rec:containsAllergen/rec:name ?a ; rec:name ?recipe  FILTER(lang(?recipe) = "en") }
GROUP BY ?recipe ORDER BY ?recipe''', reasoning=True)

,recipe,allergens
0,Chicken Stir Fry,"Sesame, Soybeans, Gluten"
1,Chocolate Brownies,"Eggs, Tree Nuts, Gluten, Milk"
2,Crispy Chicken Tenders,"Eggs, Gluten"
3,Grilled Salmon,Fish
4,Lupin Flatbread,Lupin
5,Margherita Pizza,"Gluten, Milk"
6,Moules Marinieres,"Molluscs, Sulphites, Milk"
7,Pad Thai,"Peanuts, Crustaceans, Eggs, Fish, Soybeans"
8,Pancakes,"Eggs, Gluten, Milk"
9,Pesto Pasta,"Tree Nuts, Gluten, Milk"


Chapter 5 **stored** these links with `INSERT … WHERE`, and they'd go stale when ingredients change. Here they're **derived** on every query, so they're always current.

The recursive chain `rec:underCategory` gives each ingredient its category **and all categories above it**:

In [9]:
q('SELECT ?category { ingredient:shrimp rec:underCategory/rec:name ?category }', reasoning=True)

,category
0,Protein
1,Seafood
2,Shellfish
3,Food


### Defined classes

`rec:DairyRecipe` and `rec:SeafoodRecipe` are **defined** by conditions (Chapter 7). The reasoner **classifies** recipes into them:

In [10]:
for cls in ('DairyRecipe', 'SeafoodRecipe'):
    r = both(f'SELECT ?recipe {{ ?r a rec:{cls} ; rec:name ?recipe  FILTER(lang(?recipe) = "en") }} ORDER BY ?recipe')
    print(f'{cls:14} without: {r["without reasoning"].recipe.tolist()}')
    print(f'{"":14} with:    {r["with reasoning"].recipe.tolist()}')

DairyRecipe    without: []
               with:    ['Chocolate Brownies', 'Margherita Pizza', 'Moules Marinieres', 'Pancakes', 'Pesto Pasta', 'Spaghetti Carbonara']


SeafoodRecipe  without: []
               with:    ['Grilled Salmon', 'Moules Marinieres', 'Pad Thai']


`SeafoodRecipe` shows axioms working together. Pad Thai has **shrimp**, whose category is **shellfish**; the recursive chain puts shrimp *under* **seafood**; and the class definition then matches.

## 8.4 Stardog rules

OWL describes **what things are**. It can't compare numbers or compute values. **Rules** can:

```
IF   { pattern }          ← when this matches …
THEN { triples }          ← … these triples are inferred
```

Rules live in the model graph, written in Turtle as `rule:SPARQLRule` resources:

In [11]:
print((ROOT / 'ontology' / 'recipes_rules.ttl').read_text(encoding='utf-8'))

# Stardog rules for the Recipes ontology (Chapter 8).
# Each rule: IF { pattern } THEN { new triples }. Load into the same graph as the ontology.
#
# Rule bodies may use triple patterns, FILTER, BIND and sequence paths (a/b), including
# properties that are themselves inferred. In our tests, rules containing NOT EXISTS or
# the path operators * and + produced no results: keep rule bodies positive and simple.

@prefix rule: <tag:stardog:api:rule:> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

[] a rule:SPARQLRule ;
   rdfs:label "Quick recipe: ready in 20 minutes or less" ;
   rule:content """
     PREFIX rec: <http://example.org/recipes#>
     IF   { ?r a rec:Recipe ; rec:prepMinutes ?m . FILTER(?m <= 20) }
     THEN { ?r a rec:QuickRecipe . }
   """ .

[] a rule:SPARQLRule ;
   rdfs:label "High-protein ingredient: at least 20 g protein per 100 g" ;
   rule:content """
     PREFIX rec: <http://example.org/recipes#>
     IF   { ?i a rec:Ingredient ; rec:proteinPer100g ?p . 

### A threshold

In [12]:
q('SELECT ?recipe ?minutes { ?r a rec:QuickRecipe ; rec:name ?recipe ; rec:prepMinutes ?minutes  FILTER(lang(?recipe) = "en") } ORDER BY ?minutes',
  reasoning=True)

,recipe,minutes
0,Pesto Pasta,15
1,Pancakes,20
2,Spaghetti Carbonara,20
3,Grilled Salmon,20
4,Lupin Flatbread,20


### Rules that build on rules

`HighProteinRecipe` uses `HighProteinIngredient`, which is itself inferred by another rule. Rules can also use facts inferred by OWL, and the other way round:

In [13]:
q('''SELECT ?recipe (GROUP_CONCAT(?ingredient; separator=", ") AS ?high_protein_ingredients)
WHERE { ?r a rec:HighProteinRecipe ; rec:name ?recipe ; rec:hasIngredient ?i  FILTER(lang(?recipe) = "en")
        ?i a rec:HighProteinIngredient ; rec:name ?ingredient }
GROUP BY ?recipe ORDER BY ?recipe''', reasoning=True)

,recipe,high_protein_ingredients
0,Chicken Stir Fry,Chicken Breast
1,Chocolate Brownies,"Almonds, Cocoa Powder"
2,Crispy Chicken Tenders,Chicken Breast
3,Grilled Salmon,Salmon
4,Lupin Flatbread,Lupin Flour
5,Margherita Pizza,Mozzarella
6,Pad Thai,"Peanuts, Shrimp"
7,Pesto Pasta,Parmesan Cheese
8,Spaghetti Carbonara,Parmesan Cheese
9,Thai Green Curry,Chicken Breast


### Computed values

A rule's `THEN` can create **new values** with `BIND`:

In [14]:
q('SELECT ?speed (GROUP_CONCAT(?recipe; separator=", ") AS ?recipes) { ?r rec:speed ?speed ; rec:name ?recipe  FILTER(lang(?recipe) = "en") } GROUP BY ?speed',
  reasoning=True)

,speed,recipes
0,normal,"Chicken Stir Fry, Crispy Chicken Tenders, Margherita Pizza, Moules Marinieres, Pad Tha..."
1,slow,"Chocolate Brownies, Thai Green Curry, Vegetable Soup"
2,quick,"Pesto Pasta, Grilled Salmon, Lupin Flatbread, Pancakes, Spaghetti Carbonara"


### What rules can't do: "vegan"

A vegan recipe is one with **no** animal ingredient. The natural rule would be:

```
IF   { ?r a rec:Recipe . FILTER NOT EXISTS { ?r rec:hasIngredient/rec:underCategory category:meat } }
THEN { ?r a rec:VeganRecipe . }
```

In our tests on this Stardog version, rules containing `NOT EXISTS`, and rules using the path operators `*` or `+`, **silently produced nothing**. That's no accident. Stardog's reasoning, like OWL, is **monotonic**: adding data can only **add** conclusions, never take them back. A rule saying "vegan, because nothing says otherwise" would have to be retracted the moment someone records an egg.

"No animal ingredient" is a **closed-world** question about the data as it stands, so it belongs in a **query**. Reasoning still helps: the query uses the inferred `underCategory` and `containsAllergen`:

In [15]:
print(sparql.load_query('recipes/vegan_recipes'))
sparql.run_query(conn, 'recipes/vegan_recipes', graphs=DATA, reasoning=True, schema=MODEL_NAME)

# Recipes with no meat, seafood, dairy or egg ingredient, and no animal allergen. Needs reasoning.
# Parameters: none (run with reasoning=True and the recipes model)
PREFIX rec:      <http://example.org/recipes#>
PREFIX category: <http://example.org/recipes/category/>
PREFIX allergen: <http://example.org/recipes/allergen/>

SELECT ?name
WHERE {
  ?recipe a rec:Recipe ; rec:name ?name .
  FILTER(lang(?name) = "en")
  # rec:underCategory is inferred: an ingredient's category and every category above it
  FILTER NOT EXISTS {
    ?recipe rec:hasIngredient/rec:underCategory ?c .
    VALUES ?c { category:meat category:seafood category:dairy category:eggs }
  }
  # rec:containsAllergen is inferred through the property chain
  FILTER NOT EXISTS {
    ?recipe rec:containsAllergen ?a .
    VALUES ?a { allergen:milk allergen:eggs allergen:fish allergen:crustaceans allergen:molluscs }
  }
}
ORDER BY ?name



,name
0,Lupin Flatbread
1,Vegetable Soup


Potato Salad has no ingredient in an animal category, but its **mayonnaise** carries the Eggs allergen, so the second `NOT EXISTS` excludes it. Without reasoning, neither `underCategory` nor `containsAllergen` exists, and **every** recipe would pass as vegan. That's a dangerous silent failure. Run such queries with reasoning.

| Need | Tool |
|---|---|
| What something **is**, from its properties (definitions, hierarchies, inverses, chains) | OWL axioms |
| Thresholds, arithmetic, computed labels | Stardog rules |
| "No …", "only …", counting, "exactly one" | Queries with `NOT EXISTS` / aggregates, or SHACL (Chapter 9) |

## 8.5 Explaining inferences

Inferred facts are only useful if people **trust** them. Stardog can explain every inference as a **proof**: the axioms and asserted facts it came from.

pystardog's `explain_inference()` doesn't pass a model name, so it only sees the default model. `kg.reasoning.explain()` calls the same endpoint with `schema=…`, and `format_proof()` prints the proof as a tree:

In [16]:
proof = reasoning.explain(conn, RECIPE['pad-thai'], REC.containsAllergen, ALLERGEN['peanuts'], MODEL_NAME)
print(reasoning.format_proof(proof))

INFERRED: recipe:pad-thai rec:containsAllergen allergen:peanuts
    ASSERTED: rec:containsAllergen owl:propertyChainAxiom ( rec:hasIngredient rec:hasAllergen )
    ASSERTED: ingredient:peanuts rec:hasAllergen allergen:peanuts
    ASSERTED: recipe:pad-thai rec:hasIngredient ingredient:peanuts


Read it from the top. The **inferred** fact rests on the chain axiom from the ontology and two **asserted** triples from the data. A more complex one, a rule built on a rule:

In [17]:
print(reasoning.format_proof(reasoning.explain(conn, RECIPE['pesto-pasta'], RDF.type, REC.HighProteinRecipe, MODEL_NAME)))

INFERRED: recipe:pesto-pasta rdf:type rec:HighProteinRecipe
    ASSERTED: (a Stardog rule, see ontology/recipes_rules.ttl)
    ASSERTED: recipe:pesto-pasta rec:hasIngredient ingredient:parmesan-cheese
    INFERRED: ingredient:parmesan-cheese rdf:type rec:HighProteinIngredient
        ASSERTED: (a Stardog rule, see ontology/recipes_rules.ttl)
        ASSERTED: ingredient:parmesan-cheese rec:proteinPer100g "38.0"^^xsd:decimal
        INFERRED: ingredient:parmesan-cheese rdf:type rec:Ingredient
            ASSERTED: rec:hasIngredient rdfs:range rec:Ingredient
            ASSERTED: recipe:pesto-pasta rec:hasIngredient ingredient:parmesan-cheese
    INFERRED: recipe:pesto-pasta rdf:type rec:Recipe
        ASSERTED: rec:hasIngredient rdfs:domain rec:Recipe


Pesto Pasta is high-protein **because** it has Parmesan, which is a high-protein ingredient **because** of its 38 g protein, via two rules. Along the way the proof also shows the RDFS inferences (`rdfs:domain`, `rdfs:range`) that typed the recipe and the ingredient.

A defined class:

In [18]:
print(reasoning.format_proof(reasoning.explain(conn, RECIPE['pancakes'], RDF.type, REC.DairyRecipe, MODEL_NAME)))

INFERRED: recipe:pancakes rdf:type rec:DairyRecipe
    ASSERTED: rec:DairyRecipe owl:equivalentClass […]
    ASSERTED: recipe:pancakes rec:hasIngredient ingredient:butter
    INFERRED: recipe:pancakes rdf:type rec:Recipe
        ASSERTED: rec:hasIngredient rdfs:domain rec:Recipe
        ASSERTED: recipe:pancakes rec:hasIngredient ingredient:butter


`[…]` stands for the class definition (an `owl:Restriction`). Pancakes also contain milk, which is a second, alternative proof; `format_proof` shows the first one.

## 8.6 Consistency

A few axioms can be **violated**: disjoint classes, functional properties with two different values, and so on. Data that violates them is **inconsistent**, and a reasoner can derive *anything* from a contradiction. Reasoning results from an inconsistent graph can't be trusted.

In [19]:
print('Consistent now?', reasoning.is_consistent(conn, MODEL_NAME, DATA))

conn.update(PREFIXES + f'INSERT DATA {{ GRAPH <{DATA}> {{ ingredient:egg a rec:Recipe . }} }}')   # an ingredient typed as a recipe
print('After typing an ingredient as a recipe?', reasoning.is_consistent(conn, MODEL_NAME, DATA))
print()
print(reasoning.format_proof(reasoning.explain_inconsistency(conn, MODEL_NAME, DATA)))

Consistent now? True


After typing an ingredient as a recipe? False



VIOLATED: owl:AllDisjointClasses ( rec:Recipe rec:Ingredient rec:IngredientLine rec:Category rec:Allergen rec:User rec:Rating )
    ASSERTED: owl:AllDisjointClasses ( rec:Recipe rec:Ingredient rec:IngredientLine rec:Category rec:Allergen rec:User rec:Rating )
    ASSERTED: ingredient:egg rdf:type rec:Ingredient
    ASSERTED: ingredient:egg rdf:type rec:Recipe


The explanation points at the `owl:AllDisjointClasses` axiom and the two conflicting type triples. Fix the data:

In [20]:
conn.update(PREFIXES + f'DELETE DATA {{ GRAPH <{DATA}> {{ ingredient:egg a rec:Recipe . }} }}')
print('Consistent again?', reasoning.is_consistent(conn, MODEL_NAME, DATA))

Consistent again? True


Consistency checking catches only what the **axioms** forbid. It won't notice a missing name or a negative prep time. Chapter 9's SHACL does that.

## 8.7 Reasoning in transactions and in Studio

- **Per query** (`reasoning=True, schema=…`) is what we've used, and what applications usually do.
- **Per transaction:** `conn.begin(reasoning=True)` makes queries inside that transaction reason.
- **Studio and Explorer:** the Reasoning toggle plus the model selector.

## 8.8 What reasoning costs

Reasoning rewrites queries into **bigger** queries. The query plan shows it. Asking for dairy recipes:

In [21]:
DAIRY = PREFIXES + 'SELECT ?r { ?r a rec:DairyRecipe }'
print('--- without reasoning ---')
print(reasoning.query_plan(conn, DAIRY, DATA))
print('--- with reasoning ---')
print(reasoning.query_plan(conn, DAIRY, DATA, reasoning=True, schema=MODEL_NAME))

--- without reasoning ---



# reused
prefix rec: <http://example.org/recipes#>
prefix recipe: <http://example.org/recipes/recipe/>
prefix ingredient: <http://example.org/recipes/ingredient/>
prefix category: <http://example.org/recipes/category/>
prefix allergen: <http://example.org/recipes/allergen/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix skos: <http://www.w3.org/2004/02/skos/core#>

From <urn:tutorial:ch08:data>
Projection(?r) [#1]
`─ Scan[POSC](?r, <http://www.w3.org/1999/02/22-rdf-syntax-ns#type>, rec:DairyRecipe) [#1]

--- with reasoning ---


prefix rec: <http://example.org/recipes#>
prefix recipe: <http://example.org/recipes/recipe/>
prefix ingredient: <http://example.org/recipes/ingredient/>
prefix category: <http://example.org/recipes/category/>
prefix allergen: <http://example.org/recipes/allergen/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix skos: <http://www.w3.org/2004/02/skos/core#>

From <urn:tutorial:ch08:data>
Distinct [#1]
`─ Projection(?r) [#1]
   `─ MergeJoin(?_anon_2) [#1]
      +─ Scan[POSC](?_anon_2, rec:hasAllergen, allergen:milk) [#1]
      `─ Scan[POSC](?r, rec:hasIngredient, ?_anon_2) [#1]



Without reasoning it's one index scan, which finds nothing. With reasoning, Stardog replaced "is a DairyRecipe" by the class **definition**: a join of `hasIngredient` with `hasAllergen allergen:milk`. That's exactly the query you'd write by hand.

Timings, with each query run several times:

In [22]:
def timed(query, runs=5, **kwargs):
    q(query, **kwargs)                                 # warm-up
    start = time.perf_counter()
    for _ in range(runs):
        n = len(q(query, **kwargs))
    return (time.perf_counter() - start) / runs * 1000, n

cases = [
    ('allergens: stored path, no reasoning', 'SELECT ?r ?a { ?r rec:hasIngredient/rec:hasAllergen ?a }', False),
    ('allergens: chain, reasoning',          'SELECT ?r ?a { ?r rec:containsAllergen ?a }', True),
    ('dairy: hand-written, no reasoning',    'SELECT DISTINCT ?r { ?r rec:hasIngredient/rec:hasAllergen allergen:milk }', False),
    ('dairy: defined class, reasoning',      'SELECT ?r { ?r a rec:DairyRecipe }', True),
    ('all types of all things, reasoning',   'SELECT ?s ?type { ?s a ?type }', True),
]
rows = []
for label, query, reason in cases:
    ms, n = timed(query, reasoning=reason)
    rows.append((label, n, round(ms)))
pd.DataFrame(rows, columns=['query', 'rows', 'ms per query'])

,query,rows,ms per query
0,"allergens: stored path, no reasoning",40,314
1,"allergens: chain, reasoning",36,295
2,"dairy: hand-written, no reasoning",6,297
3,"dairy: defined class, reasoning",6,261
4,"all types of all things, reasoning",341,331


On data this small, every query takes a few hundred milliseconds, and almost all of it is the round trip to Stardog Cloud. The timings don't separate the approaches here; the **plans** are what matter:

- A **targeted** reasoning query (one class, one property) is rewritten into roughly the query you'd write by hand, so it costs about the same.
- An **open-ended** question ("all types of everything") must be rewritten against **every** axiom in the model. On large data and big models, that's where reasoning gets expensive.

Note the row counts too: the stored path returns **40** rows and the chain **36**. The path gives one row per *ingredient*, so Crispy Chicken Tenders gets Gluten twice (flour and bread crumbs), while the inferred `containsAllergen` triples are a set.

**Keeping reasoning fast**

1. Ask **specific** questions: fix the class or the property.
2. Keep the **model small** and focused. Each named model only includes the axioms it needs; you choose one per query.
3. Avoid **over-general axioms**, such as domains on widely used properties or huge transitive hierarchies.
4. **Materialize** hot, stable inferences with `INSERT … WHERE` (Chapter 5) and refresh them when the data changes.
5. Measure with plans and timings. Chapter 15 goes deeper.

> 🔁 **Neo4j equivalent:** Neo4j has no reasoner. You would write every inference as a query (`MATCH (r)-[:HAS_INGREDIENT]->()-[:HAS_ALLERGEN]->(a) …`), or maintain derived relationships yourself, as in Chapter 5's materialization.

## 8.9 Exercises

**Exercise 1.** Add a rule `rec:LongRecipe` for recipes over 30 minutes. Load it into the model graph and list them with reasoning. Remove the rule afterwards.

<details><summary>Solution</summary>

```python
RULE = '''@prefix rule: <tag:stardog:api:rule:> .
<urn:tutorial:ch08:rule:long> a rule:SPARQLRule ; rule:content \"\"\"
  PREFIX rec: <http://example.org/recipes#>
  IF { ?r a rec:Recipe ; rec:prepMinutes ?m . FILTER(?m > 30) } THEN { ?r a rec:LongRecipe . } \"\"\" .'''
with client.transaction(conn):
    conn.add(stardog.content.Raw(RULE.encode(), 'text/turtle'), graph_uri=MODEL)
print(q('SELECT ?r { ?r a rec:LongRecipe }', reasoning=True))
conn.update(f'DELETE WHERE {{ GRAPH <{MODEL}> {{ <urn:tutorial:ch08:rule:long> ?p ?o }} }}')
```
Giving the rule an IRI instead of a blank node makes it easy to delete later.
</details>

**Exercise 2.** Explain why Moules Marinières is a `rec:SeafoodRecipe`.

<details><summary>Solution</summary>

```python
print(reasoning.format_proof(reasoning.explain(conn, RECIPE['moules-marinieres'], RDF.type, REC.SeafoodRecipe, MODEL_NAME)))
```
It follows from the class definition plus the mussels ingredient. The `underCategory` step through the category hierarchy is inferred too (mussels → shellfish → seafood).
</details>

**Exercise 3.** Give `recipe:pad-thai` a second prep time of 30 (as well as 25). Is the data still consistent? Why? Remove the triple afterwards.

<details><summary>Solution</summary>

```python
conn.update(PREFIXES + f'INSERT DATA {{ GRAPH <{DATA}> {{ recipe:pad-thai rec:prepMinutes 30 . }} }}')
print(reasoning.is_consistent(conn, MODEL_NAME, DATA))
conn.update(PREFIXES + f'DELETE DATA {{ GRAPH <{DATA}> {{ recipe:pad-thai rec:prepMinutes 30 . }} }}')
```
`rec:prepMinutes` is an `owl:FunctionalProperty`, so two different integers can't both be *the* value. Check what your Stardog version reports: support for functional datatype properties in consistency checks varies. Either way, **SHACL** (Chapter 9) is the reliable tool for "exactly one value".
</details>

**Exercise 4.** Write a query for "recipes that are **both** quick and dairy", using the two inferred classes.

<details><summary>Solution</summary>

```python
q('SELECT ?recipe { ?r a rec:QuickRecipe , rec:DairyRecipe ; rec:name ?recipe FILTER(lang(?recipe) = "en") }', reasoning=True)
```
</details>

In [23]:
# your answers here

## 8.10 Clean up

Removes this chapter's graphs and the `RecipeKG` model entry. Set `CLEAN_UP = False` to explore in Studio or Explorer first (model **RecipeKG**, Reasoning on).

In [24]:
CLEAN_UP = True

if CLEAN_UP:
    models.unregister_model(MODEL_NAME)
    with client.transaction(conn):
        for g in (DATA, MODEL):
            conn.clear(graph_uri=g)
    print('Removed', DATA, 'and', MODEL, '| models now:', list(models.list_models()))
conn.close()

Removed urn:tutorial:ch08:data and urn:tutorial:ch08:model | models now: ['Recipe_Basic', 'Recipe_Python', 'SupplyChain']


## Summary

- Stardog reasons **at query time** by **rewriting** queries; nothing inferred is stored. Switch it on per query with `reasoning=True, schema=<model>`.
- **RDFS** infers types from sub-classes, domains and ranges, and labels from sub-properties. Wrong domains cause wrong types, silently.
- **OWL** adds inverse and transitive properties, **property chains** (even recursive ones) and **defined classes**, which classify data automatically.
- **Rules** (`IF … THEN …`) handle thresholds, computed values and multi-step derivations. They're **monotonic**, like OWL: no "NOT EXISTS". Closed-world questions such as "vegan" are queries, helped by inferred facts.
- **Explanations** (`kg.reasoning.explain` + `format_proof`) show the proof behind any inferred fact.
- **Consistency** checks catch contradictions with the axioms (e.g. disjoint classes). Reasoning over inconsistent data can't be trusted.
- Reasoning **costs** grow with open-ended questions and big models. Ask targeted questions, keep models focused, and materialize hot inferences.

**Next: Chapter 9, Data quality with SHACL.** Closed-world validation, "exactly one", patterns, value ranges and severities, and a validation gate that rejects a bad batch before it's loaded.